In [2]:
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


from utils.hospital_env import HospitalEnv


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup complete.")

Setup complete.


In [1]:
def generate_data(n, seed=42):
    np.random.seed(seed)
    DAY = 8 * 60  

   
    arrival_time = np.random.randint(0, DAY, n)

   
    delay = np.random.randint(0, 120, n)
    slot_time = arrival_time + delay
    slot_time = np.clip(slot_time, 0, DAY - 1)

    
    priority = np.random.choice([0, 1], n, p=[0.8, 0.2])

   
    no_show_prob = []
    for p in priority:
        if p == 1:
            no_show_prob.append(np.random.uniform(0.02, 0.15))
        else:
            no_show_prob.append(np.random.uniform(0.1, 0.4))

    no_show_prob = np.array(no_show_prob)

    
    show = (np.random.rand(n) > no_show_prob).astype(int)

    data = pd.DataFrame({
        "arrival_time": arrival_time,
        "slot_time": slot_time,
        "priority": priority,
        "no_show_prob": no_show_prob,
        "show": show
    })

    return data

In [3]:
DAY_MINUTES = 480
def normalize(state):
    a, s, p, n, icu = state
    return np.array([
        a / DAY_MINUTES,
        s / DAY_MINUTES,
        p / 2.0,
        n,
        icu
    ], dtype=np.float32)


In [4]:
STATE_DIM = 5
ACTION_DIM = 2

class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(STATE_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, ACTION_DIM)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
def train_ddqn(data, episodes=50):

    policy = DQN()
    target = DQN()
    target.load_state_dict(policy.state_dict())
    target.eval()

    optimizer = optim.Adam(policy.parameters(), lr=1e-3)
    memory = deque(maxlen=5000)

    gamma = 0.99
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.995
    batch_size = 64
    target_update = 5

    episode_rewards = []

    for ep in range(episodes):
        env = HospitalEnv(data)
        state = normalize(env.reset()).astype(np.float32)
        done = False
        total_reward = 0

        while not done:

            # Epsilon-Greedy
            if random.random() < epsilon:
                action = random.randint(0, 1)
            else:
                with torch.no_grad():
                    state_tensor = torch.from_numpy(state).float().unsqueeze(0)
                    q_vals = policy(state_tensor)
                    action = torch.argmax(q_vals, dim=1).item()

            # Environment Step
            next_state_raw, reward, done, info = env.step(action)

            if not done:
                next_state = normalize(next_state_raw).astype(np.float32)
            else:
                next_state = np.zeros(STATE_DIM, dtype=np.float32)

            memory.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward

            # Training Step
            if len(memory) >= batch_size:
                batch = random.sample(memory, batch_size)
                s, a, r, ns, d = zip(*batch)

                s  = torch.from_numpy(np.stack(s)).float()
                ns = torch.from_numpy(np.stack(ns)).float()
                a  = torch.tensor(a, dtype=torch.long)
                r  = torch.tensor(r, dtype=torch.float32)
                d  = torch.tensor(d, dtype=torch.float32)

                q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()

                # Double DQN target
                with torch.no_grad():
                    next_actions = policy(ns).argmax(1)
                    q_next = target(ns).gather(1, next_actions.unsqueeze(1)).squeeze()
                    q_target = r + gamma * q_next * (1 - d)

                loss = nn.MSELoss()(q_pred, q_target)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()

        # Target Network Update
        if (ep + 1) % target_update == 0:
            target.load_state_dict(policy.state_dict())

        # Epsilon Decay
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        episode_rewards.append(total_reward)
        print(f"Episode {ep+1}/{episodes} Reward: {total_reward}")

    # Save Model
    os.makedirs("models", exist_ok=True)
    torch.save(policy.state_dict(), "models/ddqn_model.pt")

    # Return average reward for scaling experiment
    return np.mean(episode_rewards[-10:])

In [6]:
def evaluate(model, env):
    state = normalize(env.reset()).astype(np.float32)
    done = False
    total_reward = 0

    model.eval()

    while not done:
        with torch.no_grad():
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            action = torch.argmax(model(state_tensor), dim=1).item()

        next_state, reward, done, info = env.step(action)
        total_reward += reward

        if not done:
            state = normalize(next_state).astype(np.float32)

    return total_reward

In [ ]:
sizes = [500, 2000, 5000, 10000]
results = {}

for size in sizes:
    print(f"\nTraining on dataset size: {size}")

    data = generate_data(size)
    data.to_csv(os.path.join(DATA_DIR, f"synthetic_hospital_data_{size}.csv"), index=False)

    reward = train_ddqn(data)
    results[size] = reward

plt.figure()
plt.plot(list(results.keys()), list(results.values()), marker='o')
plt.xlabel("Dataset Size")
plt.ylabel("Average Reward")
plt.title("Phase 9: Dataset Scaling & Generalization")
plt.savefig("scaling_results.png")
plt.show()


Training on dataset size: 500
Episode 1/50 Reward: -1877
Episode 2/50 Reward: -2411
Episode 3/50 Reward: -1348
Episode 4/50 Reward: -1770
Episode 5/50 Reward: -1476
Episode 6/50 Reward: -1937
Episode 7/50 Reward: -1569
Episode 8/50 Reward: -1893
Episode 9/50 Reward: -1578
Episode 10/50 Reward: -1052
Episode 11/50 Reward: -2070
Episode 12/50 Reward: -1509
Episode 13/50 Reward: -1708
Episode 14/50 Reward: -1556
Episode 15/50 Reward: -1849
Episode 16/50 Reward: -1508
Episode 17/50 Reward: -957
Episode 18/50 Reward: -1542
Episode 19/50 Reward: -1808
Episode 20/50 Reward: -1974
Episode 21/50 Reward: -1818
Episode 22/50 Reward: -1596
